# BAFU import

### 1. Import required packages

In [9]:
import bw2data as bd
import bw2io as bi
import pandas as pd

from functions.fun_bafu_import import (
    apply_elementary_flow_mapping,
    load_bafu_ecospold,
    load_elementary_flow_mapping,
)
from functions.utils import Config

### 2. Select project

In [ ]:
bd.projects.set_current(Config.PROJECT_NAME)

print(f"Current Brightway project: {bd.projects.current}")

### 3. Import biosphere database

In [3]:
if "biosphere3" not in bd.databases:
    bi.create_default_biosphere3()

### 4. Convert the BAFU EcoSpold files

BAFU 2025 is provided in EcoSpold 1 format. The helper function below reads the XML files and converts them into Brightway-style dataset dictionaries.

The detailed parsing code, including unit conversion, uncertainty handling, metadata extraction, and exchange classification, is available in `functions/fun_bafu_import.py`.

In [4]:
all_data = load_bafu_ecospold(
    Config.BAFU_ECOSPOLD_PATH,
    Config.BAFU_DATABASE_NAME,
)

Reference index contains 11947 entries.
Built 11947 dataset dictionaries.


### 5. Map BAFU elementary flows

The supplied mapping file translates BAFU elementary-flow names and categories to the corresponding names and categories used by the Brightway biosphere database.

In [5]:
mapping = load_elementary_flow_mapping(Config.BAFU_MAPPING_FILE_PATH)
all_data = apply_elementary_flow_mapping(all_data, mapping)

1004 elementary-flow mapping entries loaded.
Applied elementary-flow mapping to 100498 biosphere exchanges.


### 6. Create Brightway importer

The converted BAFU datasets are assigned to a Brightway LCI importer and the standard import strategies are applied.

In [6]:
importer = bi.importers.base_lci.LCIImporter(
    db_name=Config.BAFU_DATABASE_NAME
)
importer.data = all_data
importer.apply_strategies()


Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: assign_only_product_as_production
Applying strategy: strip_biosphere_exc_locations
Applied 4 strategies in 0.40 seconds


### 7. Link technosphere and biosphere exchanges

Technosphere exchanges are linked to other BAFU activities using activity name, reference product, and location. Biosphere exchanges are linked to the configured BAFU biosphere database using name and categories.

In [7]:
importer.match_database(
    fields=["name", "reference product", "location"]
)
importer.match_database(
    Config.BAFU_BIOSPHERE_NAME,
    fields=["name", "categories"],
)

Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


### 8. Inspect import statistics

Review any unlinked exchanges before writing the database.

In [10]:
# Inspect unlinked exchanges
pd.DataFrame(importer.unlinked)

,name,unit,type,categories,comment,flow,uncertainty type,amount,loc,scale,negative,location,reference product
0,Black carbon,kilogram,biosphere,"(air,)","(1; 5; 5; 3; 4; 4); (1,5,5,3,4,4,BU:3); based ...",40586,2,2.190000e-08,-17.636779,1.691013,False,NaN,NaN
1,Ethene,kilogram,biosphere,"(air, non-urban air or from high stacks)","Uncertainty from pedigree matrix: (2, 2, 1, 2,...",649,2,2.300000e-07,-15.285187,0.036236,False,NaN,NaN
2,Ethene,kilogram,biosphere,"(air, urban air close to ground)","Uncertainty from pedigree matrix: (2, 2, 1, 2,...",649,2,2.970000e-07,-15.029534,0.036236,False,NaN,NaN
3,"Noise, road, passenger car, average",kilometer,biosphere,"(non material emissions,)","Uncertainty from pedigree matrix: (5, 5, 2, 1,...",59935,2,1.000000e+00,0.000000,0.314706,False,NaN,NaN
4,Sodium,kilogram,biosphere,"(water, surface water)",uncertainty is calculated from uncertainties i...,60865,2,3.405900e-02,-3.379661,0.395522,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
246,Dipropylthiocarbamic acid S-ethyl ester,kilogram,biosphere,"(soil, agricultural)","(2,1,1,1,1,na)",154343,2,1.622900e-05,-11.028711,0.094359,False,NaN,NaN
247,Metam-sodium dihydrate,kilogram,biosphere,"(soil, agricultural)","(2,1,1,1,1,na)",154709,2,6.697200e-04,-7.308651,0.094359,False,NaN,NaN
248,"Occupation, forest, extensive",square meter-year,biosphere,"(resources,)",NaN,98859,0,4.233316e-01,0.423332,NaN,NaN,NaN,NaN
249,"Water, embodied in product, PE",kilogram,biosphere,"(economic, primary production factor)","(4,1,1,1,1,5,BU:1.05);",81403,2,9.300000e-01,-0.072571,0.131221,False,NaN,NaN


### 9. Handle remaining unlinked exchanges

In [11]:
importer.drop_unlinked(i_am_reckless=True)

Applying strategy: drop_unlinked
Applied 1 strategies in 0.15 seconds


### 10. Write the BAFU database

This writes the processed datasets to the current Brightway project.

In [12]:
importer.write_database()

16:21:54+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 11947/11947 [00:24<00:00, 496.99it/s]


16:22:21+0200 [info     ] Vacuuming database            
Created database: bafu


Brightway2 SQLiteBackend: bafu

### 11. Verify the import

The BAFU database should now appear in the list of databases in the current Brightway project.

In [13]:
bd.databases

Databases dictionary with 4 object(s):
	bafu
	biosphere3
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-consequential